# PrecisionSkin - Lesion detector v3: train with YOUR OWN photos (machine-drafted boxes + Normal backgrounds)

Goal: improve the lesion detector with the photos you already use for the classifier (all from public datasets), **without you boxing thousands of images**.

**The idea**
1. The current detector (v1) drafts boxes on photos of Acne / Hyperpigmentation / Eczema that were **not** in the public box datasets.
2. Only **safe** drafts are used for training: confident boxes of the right condition; photos where the detector is unsure, or finds nothing, are **left out** (so real lesions are never taught as "background").
3. About 1,000 **Normal** photos with **no boxes** are added so the model learns that healthy skin has no lesions.
4. **You hand-box a small test set** (about 40 photos per condition, ~1 hour). A model cannot grade itself, so these human boxes are the honest measuring stick. They are never used for training.
5. A new detector (v3) is trained and compared with v1 on (a) your hand-boxed photos and (b) the original 566 test photos. **v1 stays in the app unless v3 is clearly better.**

**Safety:** your original photos and the v1 dataset are never changed (copy / link only). Nothing goes into the app automatically.
Kernel: the WSL **`.venv_det`** (PyTorch + ultralytics, uses the GPU). Run cells in order; read the Markdown above each cell.

`MODE = "machine"` (default): machine boxes only. `MODE = "review"`: the drafts are put in review folders so you can correct them with `eczema_review_tool.py` (optional, slower, usually better).

## 00 - Settings  (edit this cell only)

In [ ]:
import os, sys, json, math, time, random, shutil, hashlib, datetime, platform
from pathlib import Path
import numpy as np, pandas as pd, cv2, yaml
from PIL import Image

EXPERIMENT = "own_images_v3"
MODE = "machine"                    # "machine" = use safe machine boxes directly | "review" = you correct the drafts first
N_TRAIN_PER_CLASS = 500             # photos per condition that get machine-drafted boxes
N_TEST_PER_CLASS = 40               # photos per condition that YOU hand-box (test only, never trained)
ANNOTATORS = ["Elaiza", "Francesca", "Andrea"]      # who hand-boxes (names are used in folder names; no spaces)
N_CALIBRATION_PER_CLASS = 8         # photos per condition that EVERY annotator boxes (measures how much you agree)
N_NORMAL_BACKGROUND = 1000          # Normal photos added with no boxes (training only)
CONF_DRAFT = 0.15                   # the detector reports everything above this (so we can see what it is unsure about)
KEEP_CONF = 0.35                    # a box is used for training only if its confidence is at least this
ALLOWED_TYPES = ["close_up_facial", "lesion_region_facial", "full_face", "partial_face"]    # image types worth boxing (facial)
SEED = 42

# training recipe = the one used for v1 (so the comparison is fair)
MODEL_START = "yolov8n.pt"; IMGSZ = 640; EPOCHS = 100; PATIENCE = 25; BATCH = 16; WORKERS = 2
RUN_TRAINING = True                 # set False to prepare everything and train later

HOME = Path.home() / "PrecisionSkinV1"
WORK = HOME / "detector_work"
V1_DATASET = WORK / "dataset_v1"
V1_MODEL = WORK / "runs" / "detector_v1_yolov8n_seed42" / "weights" / "best.pt"
V1_EXPORT_JSON = WORK / "export" / "lesion_detector_v1.json"
REPO = Path("/mnt/c/Users/Bautista/source/repos/SkinDevApp")
CANDIDATES = {"acne": REPO / "acne_facial_candidates.csv", "hyperpigmentation": REPO / "hyperpigmentation_facial_candidates.csv",
              "eczema": REPO / "eczema_facial_candidates.csv"}
# where your classifier photos are (the folder holding Acne / Hyperpigmentation / Eczema / Normal). The first existing one is used.
_CLASSES_CANDIDATES = [Path("/mnt/c/Users/Bautista/Downloads/PrecisionSkin/Classes"), Path("/mnt/c/Users/Bautista/Downloads/Classes")]
CLASSES_ROOT = next((p for p in _CLASSES_CANDIDATES if (p / "Normal").exists()), _CLASSES_CANDIDATES[0])
NORMAL_DIR = CLASSES_ROOT / "Normal"
OUT = HOME / "detector_own_images" / EXPERIMENT                        # selections, extras, dataset, runs (WSL disk)
REVIEW_ROOT = Path("/mnt/c/Users/Bautista/Documents/PrecisionSkin_DetectorOwn")       # folders you open with the review tool (Windows side)
HOME_DOC = HOME / "Localization_Documentation"
CLASSES = ["acne", "hyperpigmentation", "eczema"]                                  # checked against the v1 model below

for p in (OUT, REVIEW_ROOT):
    p.mkdir(parents=True, exist_ok=True)
random.seed(SEED); np.random.seed(SEED)


def to_win(p):
    s = str(p)
    return ("C:\\" + s[len("/mnt/c/"):].replace("/", "\\")) if s.startswith("/mnt/c/") else s


def md5_of(path):
    h = hashlib.md5()
    with open(path, "rb") as f:
        for blk in iter(lambda: f.read(1 << 20), b""):
            h.update(blk)
    return h.hexdigest()


import torch
from ultralytics import YOLO
print("python %s | torch %s | GPU: %s" % (platform.python_version(), torch.__version__, torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE"))
need = {"v1 model": V1_MODEL, "v1 dataset": V1_DATASET / "data.yaml", "v1 export json": V1_EXPORT_JSON, "Normal photos": NORMAL_DIR,
        **{"candidates " + k: v for k, v in CANDIDATES.items()}}
for k, v in need.items():
    print("%-22s %s" % (k, "OK" if Path(v).exists() else "MISSING  <-- fix before continuing"))
assert all(Path(v).exists() for v in need.values())
v1 = YOLO(str(V1_MODEL)); print("v1 classes:", v1.names)
assert [v1.names[i] for i in range(len(CLASSES))] == CLASSES, "class order of the v1 model differs from CLASSES"
print("MODE =", MODE, "| outputs ->", OUT, "| review folders ->", REVIEW_ROOT)

## 01 - Choose the photos
**What it does:** from your three candidate lists it takes photos that are **facial**, **not already boxed in the public datasets**, and **not duplicates** (same source photo or identical file; also not identical to anything in the v1 dataset). It picks a spread over image types (close-up / lesion region / full face / partial face). Two sets per condition: a **training pool** (machine drafts) and a **test pool** (you hand-box). It also picks the Normal photos.
**Expect:** a table of counts per condition and set. **Check:** the numbers match `N_TRAIN_PER_CLASS` / `N_TEST_PER_CLASS` (if a condition has fewer eligible photos, all are used).

In [ ]:
def stratified_order(df, seed):
    df = df.sample(frac=1.0, random_state=seed).copy()
    df["_r"] = df.groupby("image_type").cumcount() / df.groupby("image_type")["image_type"].transform("size")
    return df.sort_values("_r", kind="stable").drop(columns="_r")

# hashes of everything already in the v1 dataset (so no photo is in both)
v1_md5 = set()
for sp in ("train", "val", "test"):
    for f in (V1_DATASET / sp / "images").glob("*"):
        v1_md5.add(md5_of(f))
print("v1 dataset images hashed:", len(v1_md5))

picks, seen_keys, seen_md5 = [], set(), set(v1_md5)
summary = []
for cond, csvp in CANDIDATES.items():
    df = pd.read_csv(csvp)
    df["condition"] = cond
    df["src_path"] = [str(CLASSES_ROOT / Path(str(p)).parent.name / Path(str(p)).name) for p in df.local_path_windows]      # same folder name + file name under CLASSES_ROOT
    n0 = len(df)
    df = df[(df.boxed_already.astype(str) == "False") & (df.has_boxed_twin.astype(str) == "False")]
    df = df[df.image_type.isin(ALLOWED_TYPES)]
    df = df[df.src_path.map(lambda p: Path(p).exists())]
    df = df.drop_duplicates("source_photo_key")
    test_pool = df[df.role_for_detector.str.startswith("HAND-BOX")]
    train_pool = df[df.role_for_detector.str.startswith("pseudo-label")]
    for pool_name, pool, n in (("test", test_pool, N_TEST_PER_CLASS), ("train", train_pool, N_TRAIN_PER_CLASS)):
        chosen = []
        for r in stratified_order(pool, SEED).itertuples():
            if len(chosen) >= n:
                break
            if r.source_photo_key in seen_keys:
                continue
            h = md5_of(r.src_path)
            if h in seen_md5:
                continue
            seen_keys.add(r.source_photo_key); seen_md5.add(h)
            chosen.append(dict(condition=cond, set=pool_name, filename=r.filename, src_path=r.src_path, image_type=r.image_type, source_photo_key=r.source_photo_key, md5=h))
        picks += chosen
        summary.append(dict(condition=cond, set=pool_name, eligible=len(pool), picked=len(chosen)))
print("candidates:", pd.DataFrame(summary).to_string(index=False))
sel = pd.DataFrame(picks)
sel["pick_id"] = ["%s_%s_%04d" % (r.condition, r.set, i) for i, r in enumerate(sel.itertuples())]
sel["name"] = sel.pick_id + sel.src_path.map(lambda p: Path(p).suffix.lower())
print("\nimage types picked:"); print(pd.crosstab([sel.condition, sel.set], sel.image_type).to_string())

# Normal photos (no boxes)
nrm = [p for p in sorted(NORMAL_DIR.iterdir()) if p.suffix.lower() in (".jpg", ".jpeg", ".png", ".webp")]
random.Random(SEED).shuffle(nrm)
normal_rows = []
for p in nrm:
    if len(normal_rows) >= N_NORMAL_BACKGROUND:
        break
    try:
        w, h = Image.open(p).size
    except Exception:
        continue
    if min(w, h) < 100:
        continue
    hh = md5_of(p)
    if hh in seen_md5:
        continue
    seen_md5.add(hh); normal_rows.append(dict(src_path=str(p), md5=hh, name="normal_%04d%s" % (len(normal_rows), p.suffix.lower())))
normal_sel = pd.DataFrame(normal_rows)
print("\nNormal background photos picked:", len(normal_sel))
sel.to_csv(OUT / "selection_lesion_photos.csv", index=False); normal_sel.to_csv(OUT / "selection_normal_photos.csv", index=False)

## 02 - Split the hand-boxing among the annotators
**What it does:** shares the test photos among `ANNOTATORS`. Each person gets their own folders (`<Name>__acne`, `<Name>__hyperpigmentation`, `<Name>__eczema`). A small **calibration set** (`N_CALIBRATION_PER_CLASS` photos per condition) is given to **everyone**, so we can measure how much you agree. It also writes a ready-to-send package per person: `send_to_<Name>.zip` (their photos, the box tool and a `START_HERE.txt`).
**What each person does (about 20-30 minutes):** unzip, follow `START_HERE.txt`, box **every visible lesion tightly** with the SAME rule (press **G** in the tool), press **S** to save each photo, **X** if no lesion is visible. Then zip the `labels` folders (or the whole folder) and send them back.
Tips: wheel = zoom, Space+drag = pan, **Shift+click = stamp a box of the last size**, E = contrast, H = hide boxes, V = review sheet.
**Blind on purpose:** no machine boxes are shown, so the human boxes are independent of the detector.
**Before starting, talk for 5 minutes** and agree on the rule with 2-3 example photos (what counts as one acne spot, whether to include the red halo, how to box a pigment patch). Agreement matters more than speed.
**Expect:** a table of photos per person and condition. **Check:** every person has photos for each condition; the zips exist in `REVIEW_ROOT`.

In [ ]:
test_sel = sel[sel.set == "test"].copy()
rng_ = random.Random(SEED)
assign_rows = []
for cond in CLASSES:
    names_c = sorted(test_sel[test_sel.condition == cond].name.tolist()); rng_.shuffle(names_c)
    calib, rest = names_c[:N_CALIBRATION_PER_CLASS], names_c[N_CALIBRATION_PER_CLASS:]
    for i, nme in enumerate(rest):
        assign_rows.append(dict(condition=cond, name=nme, annotator=ANNOTATORS[i % len(ANNOTATORS)], calibration=False))
    for nme in calib:
        assign_rows.append(dict(condition=cond, name=nme, annotator="ALL", calibration=True))
assign = pd.DataFrame(assign_rows)
assign.to_csv(OUT / "test_assignments.csv", index=False); assign.to_csv(REVIEW_ROOT / "test_assignments.csv", index=False)

def folder_of(annot, cond):
    return REVIEW_ROOT / ("%s__%s" % (annot, cond))

src_of = dict(zip(test_sel.name, test_sel.src_path))
for annot in ANNOTATORS:
    for cond in CLASSES:
        d = folder_of(annot, cond)
        (d / "images").mkdir(parents=True, exist_ok=True); (d / "labels").mkdir(parents=True, exist_ok=True); (d / "classes.txt").write_text(cond + "\n")
        mine = assign[(assign.condition == cond) & ((assign.annotator == annot) | (assign.annotator == "ALL"))]
        for nme in mine.name:
            if not (d / "images" / nme).exists():
                shutil.copy2(src_of[nme], d / "images" / nme)
(REVIEW_ROOT / "_drafts").mkdir(exist_ok=True)
meta = REVIEW_ROOT / "_drafts" / "meta.json"
if not meta.exists():
    json.dump({"drafted_at": time.time(), "note": "hand-box folders created"}, open(meta, "w"))
print("photos per annotator and condition (calibration photos are counted for everyone):")
cnt = {a: {c: int(((assign.condition == c) & ((assign.annotator == a) | (assign.annotator == "ALL"))).sum()) for c in CLASSES} for a in ANNOTATORS}
print(pd.DataFrame(cnt).T.to_string())

# ready-to-send packages
import zipfile
tool_src = REPO / "lesion_box_tool.py"
for annot in ANNOTATORS:
    lines = ["HOW TO BOX YOUR PHOTOS (about 20-30 minutes)", "",
             "1. Install nothing: you only need Python 3.8 or newer (https://www.python.org/downloads/ - tick 'Add python to PATH').",
             "2. Unzip this package. Open PowerShell in the unzipped folder.",
             "3. Run these three commands, ONE AT A TIME (each opens a browser tab; press Ctrl+C in PowerShell to stop one before starting the next):", ""]
    for i, cond in enumerate(CLASSES):
        lines.append('   py -3 lesion_box_tool.py "%s__%s" --label %s --annotator %s --port %d' % (annot, cond, cond, annot, 8800 + i))
    lines += ["", "4. In the browser: draw a TIGHT box around EVERY visible lesion. S = save + next photo, X = no lesion visible, G = the rule, V = review sheet.",
              "   Mouse wheel = zoom, Space+drag = move the picture, Shift+click = stamp a box of your last size, E = contrast, H = hide boxes.",
              "5. Use the SAME rule for every photo. Agree on it with the team first.",
              "6. When finished, zip the three folders (%s__acne, %s__hyperpigmentation, %s__eczema) and send them back." % (annot, annot, annot),
              "   Do not rename files. Nothing is uploaded anywhere by the tool; it runs on your computer only."]
    zp = REVIEW_ROOT / ("send_to_%s.zip" % annot)
    with zipfile.ZipFile(zp, "w", zipfile.ZIP_DEFLATED) as z:
        z.writestr("START_HERE.txt", "\n".join(lines))
        z.write(tool_src, "lesion_box_tool.py")
        z.write(meta, "_drafts/meta.json")
        for cond in CLASSES:
            for f in folder_of(annot, cond).rglob("*"):
                if f.is_file():
                    z.write(f, str(f.relative_to(REVIEW_ROOT)))
    print("package ready:", zp, "(%.1f MB)" % (zp.stat().st_size / 1e6))
print("\nYour own commands (Windows PowerShell inside the SkinDevApp folder), e.g. for %s:" % ANNOTATORS[0])
for i, cond in enumerate(CLASSES):
    print('  py -3.13 lesion_box_tool.py "%s" --label %s --annotator %s --port %d' % (to_win(folder_of(ANNOTATORS[0], cond)), cond, ANNOTATORS[0], 8800 + i))
print("\nWhen the others send their folders back: copy them into", REVIEW_ROOT, "(replace the matching folders), then run section 06.")

## 03 - Machine drafts on the training photos
**What it does:** runs the **v1 detector** on the training-pool photos and records **every** detection above `CONF_DRAFT` (box, class, confidence) in `drafts_all.csv`.
**Expect:** a progress line and counts of how many photos have confident boxes, only unsure boxes, or nothing. **Check:** the share of photos with at least one confident box of the right class (these are the ones that can be used safely).

In [ ]:
train_sel = sel[sel.set == "train"].reset_index(drop=True)
rows = []
BATCH_PRED = 32
t0 = time.time()
for s in range(0, len(train_sel), BATCH_PRED):
    chunk = train_sel.iloc[s:s + BATCH_PRED]
    res = v1.predict([str(p) for p in chunk.src_path], imgsz=IMGSZ, conf=CONF_DRAFT, iou=0.45, max_det=100, verbose=False, device=0 if torch.cuda.is_available() else "cpu")
    for r, o in zip(chunk.itertuples(), res):
        if o.boxes is not None and len(o.boxes):
            for (x0, y0, x1, y1), c, cf in zip(o.boxes.xyxyn.cpu().numpy(), o.boxes.cls.cpu().numpy(), o.boxes.conf.cpu().numpy()):
                rows.append(dict(pick_id=r.pick_id, condition=r.condition, cls=CLASSES[int(c)], conf=float(cf), cx=float((x0 + x1) / 2), cy=float((y0 + y1) / 2), w=float(x1 - x0), h=float(y1 - y0)))
    print("  drafted %d / %d (%.0fs)" % (min(s + BATCH_PRED, len(train_sel)), len(train_sel), time.time() - t0), end="\r")
drafts = pd.DataFrame(rows, columns=["pick_id", "condition", "cls", "conf", "cx", "cy", "w", "h"])
drafts.to_csv(OUT / "drafts_all.csv", index=False)

by_pick = {k: g for k, g in drafts.groupby("pick_id")}
status = []
for r in train_sel.itertuples():
    d = by_pick.get(r.pick_id)
    own = d[d.cls == r.condition] if d is not None else None
    status.append("no_detection_of_this_condition" if own is None or len(own) == 0 else ("has_unsure_box" if (own.conf < KEEP_CONF).any() else "usable"))
train_sel["draft_status"] = status
print("\n\nmachine-draft quality per condition (training pool):")
print(pd.crosstab(train_sel.condition, train_sel.draft_status).to_string())
print("\nwhat the detector says on the same photos, other classes (info only):")
print(drafts[drafts.cls != drafts.condition].groupby(["condition", "cls"]).size().to_string() or "none")
train_sel.to_csv(OUT / "train_pool_with_status.csv", index=False)

## 04 - (optional) Review mode: correct the drafts yourself
Only if `MODE = "review"`. **What it does:** puts the training-pool photos and their drafts (condition boxes down to `CONF_DRAFT`) into `REVIEW_ROOT\train_review\<Condition>` folders so you can fix them with `lesion_box_tool.py` (same tool as in section 02). **Add every missing lesion**, delete wrong boxes, press S to save each photo. In `machine` mode this cell does nothing.

In [ ]:
if MODE != "review":
    print("MODE = 'machine': nothing to do here.")
else:
    for cond in CLASSES:
        d = REVIEW_ROOT / "train_review" / cond.capitalize()
        (d / "images").mkdir(parents=True, exist_ok=True); (d / "labels").mkdir(parents=True, exist_ok=True); (d / "classes.txt").write_text(cond + "\n")
        for r in train_sel[train_sel.condition == cond].itertuples():
            dst = d / "images" / r.name
            if not dst.exists():
                shutil.copy2(r.src_path, dst)
            lab = d / "labels" / (Path(r.name).stem + ".txt")
            if not lab.exists():
                dd = by_pick.get(r.pick_id)
                own = dd[dd.cls == cond] if dd is not None else []
                lab.write_text("".join("0 %.6f %.6f %.6f %.6f\n" % (b.cx, b.cy, b.w, b.h) for b in own.itertuples()) if len(own) else "")
    (REVIEW_ROOT / "train_review" / "_drafts").mkdir(parents=True, exist_ok=True)
    json.dump({"drafted_at": time.time(), "note": "drafts written"}, open(REVIEW_ROOT / "train_review" / "_drafts" / "meta.json", "w"))
    print("review folders ready in", REVIEW_ROOT / "train_review")
    for i, cond in enumerate(CLASSES):
        print('  py -3.13 lesion_box_tool.py "%s" --label %s --port %d' % (to_win(REVIEW_ROOT / "train_review" / cond.capitalize()), cond, 8810 + i))

## 05 - Build the extra training data
**What it does:** creates `extras/` (images + YOLO labels) from:
- **machine mode:** only photos whose draft status is `usable` (all boxes of the photo's condition are confident); other detections of other classes are not used;
- **review mode:** photos you saved in the review folders (at least one box);
- **Normal photos** with empty label files (backgrounds).
**Expect:** counts of images and boxes per condition. **Check:** enough usable photos per condition (a few hundred is good). Very few usable photos means the v1 detector is too unsure on your photos; lower `KEEP_CONF` a little or use review mode.

In [ ]:
EX = OUT / "extras"
if EX.exists():
    shutil.rmtree(EX)
(EX / "images").mkdir(parents=True); (EX / "labels").mkdir(parents=True)
cid = {c: i for i, c in enumerate(CLASSES)}
n_img, n_box = {}, {}
if MODE == "machine":
    use = train_sel[train_sel.draft_status == "usable"]
    for r in use.itertuples():
        own = by_pick[r.pick_id]; own = own[own.cls == r.condition]
        shutil.copy2(r.src_path, EX / "images" / r.name)
        (EX / "labels" / (Path(r.name).stem + ".txt")).write_text("".join("%d %.6f %.6f %.6f %.6f\n" % (cid[r.condition], b.cx, b.cy, b.w, b.h) for b in own.itertuples()))
        n_img[r.condition] = n_img.get(r.condition, 0) + 1; n_box[r.condition] = n_box.get(r.condition, 0) + len(own)
else:
    t_ref = json.load(open(REVIEW_ROOT / "train_review" / "_drafts" / "meta.json"))["drafted_at"]
    for r in train_sel.itertuples():
        lab = REVIEW_ROOT / "train_review" / r.condition.capitalize() / "labels" / (Path(r.name).stem + ".txt")
        if not lab.exists() or lab.stat().st_mtime <= t_ref + 1:
            continue
        lines = [l.split() for l in lab.read_text().splitlines() if len(l.split()) == 5]
        if not lines:
            continue
        shutil.copy2(r.src_path, EX / "images" / r.name)
        (EX / "labels" / (Path(r.name).stem + ".txt")).write_text("".join("%d %s %s %s %s\n" % (cid[r.condition], *l[1:]) for l in lines))
        n_img[r.condition] = n_img.get(r.condition, 0) + 1; n_box[r.condition] = n_box.get(r.condition, 0) + len(lines)
for r in normal_sel.itertuples():
    shutil.copy2(r.src_path, EX / "images" / r.name)
    (EX / "labels" / (Path(r.name).stem + ".txt")).write_text("")
print("extra training photos with boxes:", n_img, "| boxes:", n_box)
print("Normal background photos (no boxes):", len(normal_sel))
assert sum(n_img.values()) > 0, "no usable photos - see the note above"

## 06 - Collect and check the hand-boxed test set
**What it does:** reads the saved boxes of every annotator, builds `own_test/` (each photo uses the boxes of its assigned annotator; calibration photos use the first annotator who saved them), and prints: progress per person, a box-size summary per condition, photos with unusual boxes, and the **agreement between annotators** on the shared calibration photos (F1 at IoU 0.5).
**Expect:** progress tables, then an agreement table. **Check:** every condition has at least ~25 boxed photos; **agreement F1 about 0.6 or higher** is reasonable for lesion boxes (a low number means the rule was applied differently: discuss and re-box the calibration photos). You can re-run this cell as boxes come in.

In [ ]:
t_ref = json.load(open(REVIEW_ROOT / "_drafts" / "meta.json"))["drafted_at"]
assign = pd.read_csv(REVIEW_ROOT / "test_assignments.csv")
src_of = dict(zip(test_sel.name, test_sel.src_path))

def read_lab(p):
    return [tuple(float(v) for v in l.split()[1:]) for l in p.read_text().splitlines() if len(l.split()) == 5] if p.exists() else []

def saved_lab(annot, cond, nme):
    lab = REVIEW_ROOT / ("%s__%s" % (annot, cond)) / "labels" / (Path(nme).stem + ".txt")
    return lab if (lab.exists() and lab.stat().st_mtime > t_ref + 1) else None

OWN_TEST = OUT / "own_test"
if OWN_TEST.exists():
    shutil.rmtree(OWN_TEST)
(OWN_TEST / "images").mkdir(parents=True); (OWN_TEST / "labels").mkdir(parents=True)
rows, prog, qrows = [], [], []
for cond in CLASSES:
    photos = assign[assign.condition == cond]
    saved = used = boxes = 0
    for r in photos.itertuples():
        owners = ANNOTATORS if r.annotator == "ALL" else [r.annotator]
        got = [(a, saved_lab(a, cond, r.name)) for a in owners]
        got = [(a, l) for a, l in got if l is not None]
        if not got:
            continue
        saved += 1
        lines = read_lab(got[0][1])
        if not lines:
            continue
        used += 1; boxes += len(lines)
        shutil.copy2(src_of[r.name], OWN_TEST / "images" / r.name)
        (OWN_TEST / "labels" / (Path(r.name).stem + ".txt")).write_text("".join("%d %.6f %.6f %.6f %.6f\n" % (cid[cond], *b) for b in lines))
    rows.append(dict(condition=cond, photos=len(photos), saved=saved, used_with_boxes=used, boxes=boxes))
for annot in ANNOTATORS:
    pr = {}
    for cond in CLASSES:
        mine = assign[(assign.condition == cond) & ((assign.annotator == annot) | (assign.annotator == "ALL"))]
        pr[cond] = "%d / %d" % (sum(saved_lab(annot, cond, n) is not None for n in mine.name), len(mine))
    prog.append(dict(annotator=annot, **pr))
print("PROGRESS (saved / assigned):"); print(pd.DataFrame(prog).to_string(index=False))
print("\nTEST SET BUILT:"); print(pd.DataFrame(rows).to_string(index=False))
yaml.safe_dump({"path": str(OWN_TEST), "train": "images", "val": "images", "test": "images", "names": {i: c for i, c in enumerate(CLASSES)}}, open(OWN_TEST / "data.yaml", "w"))
HAVE_OWN_TEST = all(r["used_with_boxes"] >= 10 for r in rows)
print("\nhand-boxed test ready:", HAVE_OWN_TEST, "(needs at least 10 boxed photos per condition; 25+ is better)")

# ---- quality summary of the boxes ----
flagged = []
for lab in sorted((OWN_TEST / "labels").glob("*.txt")):
    cond = [c for c in CLASSES if lab.stem.startswith(c + "_")][0]
    w_, h_ = Image.open(list((OWN_TEST / "images").glob(lab.stem + ".*"))[0]).size
    for l in lab.read_text().splitlines():
        c_, cx, cy, bw, bh = l.split(); bw, bh = float(bw), float(bh)
        qrows.append(dict(cond=cond, image=lab.stem, w_px=bw * w_, h_px=bh * h_, area_frac=bw * bh))
if qrows:
    q = pd.DataFrame(qrows)
    print("\nbox size summary (pixels) per condition:")
    print(q.groupby("cond")[["w_px", "h_px"]].describe(percentiles=[.5]).round(0).T.to_string())
    print("boxes per photo:"); print(q.groupby(["cond", "image"]).size().groupby("cond").describe().round(1).to_string())
    for cond, g in q.groupby("cond"):
        med = g.area_frac.median()
        flagged += [(cond, i) for i in g[(g.area_frac > 6 * med) | (g.area_frac < med / 6)].image.unique()]
    print("\nphotos with an unusually large or small box (re-check with the review sheet, V):", sorted(set(flagged)) or "none")

# ---- agreement between annotators on the shared calibration photos ----
def iou_xywh(a, b):
    ax0, ay0, ax1, ay1 = a[0] - a[2] / 2, a[1] - a[3] / 2, a[0] + a[2] / 2, a[1] + a[3] / 2
    bx0, by0, bx1, by1 = b[0] - b[2] / 2, b[1] - b[3] / 2, b[0] + b[2] / 2, b[1] + b[3] / 2
    iw_, ih_ = max(0, min(ax1, bx1) - max(ax0, bx0)), max(0, min(ay1, by1) - max(ay0, by0)); i = iw_ * ih_
    u = a[2] * a[3] + b[2] * b[3] - i
    return i / u if u > 0 else 0.0

def pair_f1(A, B):
    used_, tp = set(), 0
    for x in A:
        best, bj = 0.0, None
        for j, y in enumerate(B):
            if j in used_:
                continue
            v = iou_xywh(x, y)
            if v > best:
                best, bj = v, j
        if bj is not None and best >= 0.5:
            used_.add(bj); tp += 1
    return 2 * tp / max(1, len(A) + len(B))

agree = []
for cond in CLASSES:
    calib = assign[(assign.condition == cond) & (assign.annotator == "ALL")]
    for i, a1 in enumerate(ANNOTATORS):
        for a2 in ANNOTATORS[i + 1:]:
            f1s = []
            for nme in calib.name:
                l1, l2 = saved_lab(a1, cond, nme), saved_lab(a2, cond, nme)
                if l1 is None or l2 is None:
                    continue
                A_, B_ = read_lab(l1), read_lab(l2)
                if A_ or B_:
                    f1s.append(pair_f1(A_, B_))
            if f1s:
                agree.append(dict(condition=cond, pair="%s vs %s" % (a1, a2), photos=len(f1s), agreement_F1=round(float(np.mean(f1s)), 3)))
print("\nAGREEMENT between annotators on the shared calibration photos (F1 at IoU 0.5; ~0.6+ is reasonable):")
if agree:
    ag = pd.DataFrame(agree); print(ag.to_string(index=False)); print("overall mean:", round(float(ag.agreement_F1.mean()), 3))
    ag.to_csv(OUT / "annotator_agreement.csv", index=False)
else:
    print("not available yet (the calibration photos need boxes from at least two people)")

## 07 - Build dataset v3 (v1 data + your extras, same validation and test photos)
**What it does:** creates `dataset_v3/` where the v1 train/val/test files are **linked** (no copies, v1 is untouched) and your extras are added to the **training** split only. Validation and test are exactly the v1 photos, so the old-vs-new comparison is fair.
**Expect:** counts per split. **Check:** `val` and `test` equal v1's (566 each).

In [ ]:
DS = OUT / "dataset_v3"
if DS.exists():
    shutil.rmtree(DS)
for sp in ("train", "val", "test"):
    (DS / sp / "images").mkdir(parents=True); (DS / sp / "labels").mkdir(parents=True)
    for f in (V1_DATASET / sp / "images").iterdir():
        os.symlink(f, DS / sp / "images" / f.name)
    for f in (V1_DATASET / sp / "labels").iterdir():
        os.symlink(f, DS / sp / "labels" / f.name)
for f in (EX / "images").iterdir():
    shutil.copy2(f, DS / "train" / "images" / ("own_" + f.name))
for f in (EX / "labels").iterdir():
    shutil.copy2(f, DS / "train" / "labels" / ("own_" + f.name))
yaml.safe_dump({"path": str(DS), "train": "train/images", "val": "val/images", "test": "test/images", "names": {i: c for i, c in enumerate(CLASSES)}}, open(DS / "data.yaml", "w"))
for sp in ("train", "val", "test"):
    print("%-6s images: %d" % (sp, len(list((DS / sp / "images").iterdir()))))
print("v1 for comparison: train %d | val %d | test %d" % tuple(len(list((V1_DATASET / s / "images").iterdir())) for s in ("train", "val", "test")))

## 08 - Train detector v3  (about 1.5 hours on your GPU)
**What it does:** trains YOLOv8n with the **same recipe as v1** (640 px, 100 epochs, patience 25, batch 16, seed 42) on dataset v3. The run is saved in `OUT/runs/`.
**Before you run:** close Edge/extra windows, keep the laptop plugged in and an Ubuntu window open (WSL must not shut down). **Expect:** the usual ultralytics epoch table. **Check:** validation mAP50 rises and then levels off; the run ends with early stopping or after 100 epochs.

In [ ]:
RUN_NAME = "detector_%s" % EXPERIMENT
best_pt = OUT / "runs" / RUN_NAME / "weights" / "best.pt"
if not RUN_TRAINING:
    print("RUN_TRAINING = False: skipped.")
elif best_pt.exists():
    print("already trained:", best_pt, "(delete the run folder to train again)")
else:
    m = YOLO(MODEL_START)
    m.train(data=str(DS / "data.yaml"), imgsz=IMGSZ, epochs=EPOCHS, patience=PATIENCE, batch=BATCH, device=0 if torch.cuda.is_available() else "cpu",
            workers=WORKERS, seed=SEED, cos_lr=True, close_mosaic=10, project=str(OUT / "runs"), name=RUN_NAME, exist_ok=True,
            fliplr=0.5, flipud=0.0, degrees=10, scale=0.5, hsv_h=0.015, hsv_s=0.5, hsv_v=0.35)
    print("best weights:", best_pt)

## 09 - Compare v1 and v3 on the SAME photos
**What it does:** evaluates both detectors on (a) the **original 566 test photos** (human-drawn boxes from the public datasets) and (b) **your hand-boxed photos** (if section 06 is ready). It prints precision, recall, mAP50 and mAP50-95 per class and the change.
**Expect:** two tables and a plain verdict. **Check:** v3 is only worth deploying if it is clearly better on **your** photos and not worse on the original test photos. Small differences are noise (especially with few hand-boxed photos); treat gains under about 0.02 mAP50 as "no real change".

In [ ]:
def evaluate(weights, data_yaml, split, tag):
    m = YOLO(str(weights))
    r = m.val(data=str(data_yaml), split=split, imgsz=IMGSZ, conf=0.001, plots=False, verbose=False, workers=WORKERS, batch=BATCH)
    per = {}
    for k, ci in enumerate(r.box.ap_class_index):
        p, rc, ap50, ap = r.box.class_result(k)
        per[CLASSES[int(ci)]] = dict(precision=float(p), recall=float(rc), mAP50=float(ap50), mAP50_95=float(ap))
    per["ALL"] = dict(precision=float(r.box.mp), recall=float(r.box.mr), mAP50=float(r.box.map50), mAP50_95=float(r.box.map))
    return per

def compare(data_yaml, split, title):
    a = evaluate(V1_MODEL, data_yaml, split, "v1"); b = evaluate(best_pt, data_yaml, split, "v3")
    rows = []
    for c in CLASSES + ["ALL"]:
        if c in a and c in b:
            for m_ in ("precision", "recall", "mAP50", "mAP50_95"):
                rows.append(dict(cls=c, metric=m_, v1=round(a[c][m_], 3), v3=round(b[c][m_], 3), change=round(b[c][m_] - a[c][m_], 3)))
    df = pd.DataFrame(rows)
    print("\n=== %s ===" % title)
    print(df.pivot(index="metric", columns="cls", values=["v1", "v3", "change"]).round(3).to_string())
    return df, a, b

if not best_pt.exists():
    raise RuntimeError("train v3 first (section 08)")
df_orig, a_orig, b_orig = compare(V1_DATASET / "data.yaml", "test", "ORIGINAL 566 test photos (public human-drawn boxes)")
df_orig.to_csv(OUT / "compare_original_test.csv", index=False)
if HAVE_OWN_TEST:
    df_own, a_own, b_own = compare(OWN_TEST / "data.yaml", "val", "YOUR hand-boxed photos")
    df_own.to_csv(OUT / "compare_own_test.csv", index=False)
    d_own = b_own["ALL"]["mAP50"] - a_own["ALL"]["mAP50"]; d_orig = b_orig["ALL"]["mAP50"] - a_orig["ALL"]["mAP50"]
    print("\nmAP50 change on your photos: %+.3f | on the original test photos: %+.3f" % (d_own, d_orig))
    if d_own >= 0.02 and d_orig >= -0.01:
        VERDICT = "v3 is better on your photos and not worse on the original test: worth using."
    elif d_own <= -0.02 or d_orig <= -0.03:
        VERDICT = "v3 is WORSE: keep v1."
    else:
        VERDICT = "no clear difference: keep v1 (it is already deployed and verified)."
else:
    VERDICT = "your hand-boxed set is not ready, so only the original test could be compared: do NOT replace v1 yet."
print("\nVERDICT:", VERDICT)

## 10 - Export v3 (ONNX) and save the records
**What it does:** exports `lesion_detector_v3_...onnx` + a JSON with settings and results into `detector_work/export/` (v1 files are not touched) and writes a summary for your documentation.
**Expect:** the file names and the CPU latency. **Check:** the verdict above. **Putting v3 into the app is a separate step**: only if the verdict says it is worth using, tell me and I will copy it as `lesion_detector_v1.onnx/.json`, rebuild and re-tune the thresholds on the validation set.

In [ ]:
import onnxruntime as ort
onnx_src = Path(YOLO(str(best_pt)).export(format="onnx", imgsz=IMGSZ, opset=12, simplify=True, dynamic=False))
EXP = WORK / "export"; EXP.mkdir(parents=True, exist_ok=True)
onnx_path = EXP / ("lesion_detector_v3_own_images_yolov8n_imgsz%d.onnx" % IMGSZ)
shutil.copy2(onnx_src, onnx_path)
so = ort.SessionOptions(); so.intra_op_num_threads = 4
sess = ort.InferenceSession(str(onnx_path), so, providers=["CPUExecutionProvider"])
x = np.random.rand(1, 3, IMGSZ, IMGSZ).astype("float32"); inp = sess.get_inputs()[0]
for _ in range(3):
    sess.run(None, {inp.name: x})
ts = []
for _ in range(20):
    t0 = time.perf_counter(); out = sess.run(None, {inp.name: x}); ts.append((time.perf_counter() - t0) * 1000)
lat = {"median_ms": round(float(np.median(ts)), 1), "p95_ms": round(float(np.percentile(ts, 95)), 1), "threads": 4}
meta = dict(model="PrecisionSkin lesion detector v3 (own images)", created=datetime.datetime.now().isoformat(timespec="seconds"), classes=CLASSES, imgsz=IMGSZ,
            mode=MODE, n_extra_training_photos=n_img, n_extra_boxes=n_box, n_normal_background=len(normal_sel), keep_conf=KEEP_CONF,
            hand_boxed_test=[r for r in rows], results_original_test=df_orig.to_dict("records"), results_own_test=(df_own.to_dict("records") if HAVE_OWN_TEST else None),
            verdict=VERDICT, cpu_latency=lat, onnx_sha256=hashlib.sha256(onnx_path.read_bytes()).hexdigest(),
            note="Research prototype. Machine-drafted boxes (v1 detector, confident ones only) were used for training; the hand-boxed test set was never trained on.")
json.dump(meta, open(EXP / (onnx_path.stem + ".json"), "w"), indent=2, default=str)
print("saved:", onnx_path.name, "| CPU latency:", lat)
print("\nVERDICT:", VERDICT)
shutil.copy2(EXP / (onnx_path.stem + ".json"), OUT / "v3_summary.json")
print("records in:", OUT)